# LAB 03 SOLUTION: Parquet Internals & Silver Cleaning Part 1 (Structural Hygiene)
**Course:** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Target Audience:** Tutors & Instructors (Executed Reference Solution — Case 0 EuroPack Bottling)

---

## 🎯 1. Context & Architecture-as-Code Lineage
Reference solution for **Case 0 (`EuroPack Beverage Bottling`)** bridging **Lecture 02's Apache Parquet internals** with **Silver Structural Hygiene** (`bronze_staged/*.parquet` -> `silver_part1/*.parquet`). Each operational table is handled **one file and one issue at a time** (**Problem → Discovery → Solution**); final Silver Schema Enforcement & DuckDB Audit follow in Lab 04:

```mermaid
flowchart TD
    B0["2. Load Bronze Stage (bronze_staged/*.parquet)"] --> P0["3. Parquet Internals & Pushdown Demo (3.1 - 3.4)"]
    P0 --> F1["4. File 1: bronze_telemetry.parquet (4.1 - 4.7)"]
    F1 --> F2["5. File 2: bronze_stoppages.parquet (5.1 - 5.2)"]
    F2 --> SC["6. Bronze DAMA DQ Scorecard & Save silver_part1/*.parquet (6.1 - 6.2)"]
```

## ⚙️ 2. Environment Setup & Bronze Input Discovery
### 2.1 Resolve Bronze Inputs, PyArrow/DuckDB Tooling & Load All 5 Tables

In [1]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case0_europack_bottling"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

BRONZE_STAGED = resolve_stage_dir("bronze_staged")
SILVER_P1 = SANDBOX_DATA / "silver_part1"
SILVER_P1.mkdir(parents=True, exist_ok=True)

df_telem = pd.read_parquet(BRONZE_STAGED / "bronze_telemetry.parquet")
df_stop = pd.read_parquet(BRONZE_STAGED / "bronze_stoppages.parquet")
df_lines = pd.read_parquet(BRONZE_STAGED / "bronze_lines.parquet")
df_shifts = pd.read_parquet(BRONZE_STAGED / "bronze_shifts.parquet")
df_formats = pd.read_parquet(BRONZE_STAGED / "bronze_bottle_formats.parquet")

print("Bronze source :", BRONZE_STAGED)
print("Silver Part 1 :", SILVER_P1)
print(
    f"Loaded 5 Bronze tables -> "
    f"telemetry: {len(df_telem):,} rows | stoppages: {len(df_stop):,} rows | "
    f"lines: {len(df_lines)} | shifts: {len(df_shifts)} | formats: {len(df_formats)}"
)


Bronze source : <REPO_ROOT>/05_case_studies_reference_solutions/case0_europack_bottling/data/bronze_staged
Silver Part 1 : <REPO_ROOT>/03_labs_student_starters/sandbox_outputs/case0_europack_bottling/silver_part1
Loaded 5 Bronze tables -> telemetry: 23,164 rows | stoppages: 292 rows | lines: 4 | shifts: 3 | formats: 3


## 📦 3. Inside Apache Parquet — Anatomy, Header, Footer & Pushdown (Lecture 02 Bridge)
In **Lecture 02 (§5.3)** you saw why Apache Parquet beats CSV/JSON for analytical storage. Let's open `bronze_telemetry.parquet` as raw binary bytes and via `pyarrow.parquet.ParquetFile`—**without loading the data rows**—to verify every part of its internal layout:

```text
 [0 .. 4)              Magic Header: b'PAR1' (4 bytes)
 [4 .. footer_start)   Row Groups -> Column Chunks -> Pages (Dictionary + RLE + Snappy)
 [footer_start .. -8)  File Footer Metadata (Thrift: schema, byte offsets, min/max/null stats)
 [-8 .. -4)            4-byte little-endian integer = Footer length in bytes
 [-4 .. EOF)           Magic Footer: b'PAR1' (4 bytes)
```

### 3.1 Demo: Magic Bytes (`PAR1`), Footer Length, Metadata & Embedded Schema
**Why is the File Footer placed at the end and read FIRST?** Because while writing, the engine does not know the final byte offsets or min/max statistics until the last row is flushed. When reading, Parquet jumps directly to `EOF - 8`, reads the 4-byte `footer_len`, and loads the File Footer metadata before touching a single data page!

*Run the ready-to-launch demo cell below to inspect the raw bytes and embedded Arrow schema of `bronze_telemetry.parquet`.*

In [2]:
# Ready-to-run Demo: Inspect the raw binary structure and metadata footer of a Parquet file
pq_path = BRONZE_STAGED / "bronze_telemetry.parquet"

# Read the entire file as raw bytes so we can verify its binary header/footer layout
raw_bytes = pq_path.read_bytes()

# 1. Verify the 4-byte "PAR1" magic signature at the very start and very end of the file
magic_header = raw_bytes[:4]
magic_footer = raw_bytes[-4:]

# 2. The 4 bytes right before the final "PAR1" (indices -8..-4) store the Footer length
#    as a little-endian integer (least-significant byte first, per the Parquet spec)
footer_len = int.from_bytes(raw_bytes[-8:-4], "little")

# 3. Open ONLY the Parquet metadata handle — this reads the Footer, NOT the row data pages
pf = pq.ParquetFile(pq_path)

# Sanity checks: confirm valid PAR1 markers and that our decoded footer_len matches PyArrow
assert magic_header == b"PAR1" and magic_footer == b"PAR1", "Invalid Parquet magic bytes!"
assert footer_len == pf.metadata.serialized_size, "Footer length mismatch!"

print(f"File size         : {len(raw_bytes):,} bytes")
print(f"Magic header/tail : {magic_header} ... {magic_footer}")
print(f"Footer length     : {footer_len:,} bytes (matches pf.metadata.serialized_size)")
print(f"Rows / Cols / RGs : {pf.metadata.num_rows:,} rows | {pf.metadata.num_columns} cols | {pf.metadata.num_row_groups} Row Group(s)")
print(f"Format / Writer   : v{pf.metadata.format_version} | {pf.metadata.created_by}")
print(f"Custom KV keys    : {list((pf.metadata.metadata or {}).keys())}")
print("\nEmbedded Arrow Schema (notice 'timestamp' & '_ingestion_timestamp_utc' are still STRING in Bronze!):")
print(pf.schema_arrow)


File size         : 261,026 bytes
Magic header/tail : b'PAR1' ... b'PAR1'
Footer length     : 6,985 bytes (matches pf.metadata.serialized_size)
Rows / Cols / RGs : 23,164 rows | 12 cols | 1 Row Group(s)
Format / Writer   : v2.6 | parquet-cpp-arrow version 26.0.1
Custom KV keys    : [b'ARROW:schema', b'pandas']

Embedded Arrow Schema (notice 'timestamp' & '_ingestion_timestamp_utc' are still STRING in Bronze!):
timestamp: string
line_id: string
bottle_format: string
ambient_temp_C: double
pasteurizer_temp_C: double
conveyor_speed_bpm: int64
filler_pressure_bar: double
valve_cycles_k: int64
bottled_units: int64
rejected_units: int64
_ingestion_timestamp_utc: string
_source_file: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1634


### 3.2 Demo: Column Chunks, Byte Offsets, Encodings (`Dictionary` + `RLE` + `Snappy`) & Footer Statistics
Run this demo to inspect `pf.metadata.row_group(0)` column-by-column **strictly from the metadata footer** (without reading or decompressing any data pages) and observe:
1. **Columnar Byte Layout:** Each Column Chunk starts right after the previous chunk (`dictionary_page_offset` / `data_page_offset`), and `4 + sum(compressed_B) + footer_len + 8` equals the **exact byte size of the file**!
2. **Dictionary + RLE Compression:** Look at `_source_file` and `_ingestion_timestamp_utc`: `23,164` repeated strings compress down to ~`200–265 bytes`!
3. **Free DQ Diagnostics from Footer Statistics:** Look at `nulls`, `min`, and `max`—we can already spot `120` missing temperatures, `-99.0`/`85.0 bar` filler pressure, `-120` bottled units, `'LINE_A'`..`'line_c'`, and `'2099-01-01'` *without decompressing a single row*!

In [3]:
# Ready-to-run Demo: Read per-column layout & min/max/null stats directly from Footer metadata
rg = pf.metadata.row_group(0)

chunk_rows = []
for i in range(rg.num_columns):
    # c = metadata descriptor for column i; st = precomputed min/max/null_count stats
    c = rg.column(i)
    st = c.statistics
    chunk_rows.append({
        "column": c.path_in_schema,                 # Column name in the file schema
        "phys_type": c.physical_type,               # Low-level Parquet storage type (INT64, DOUBLE, BYTE_ARRAY)
        "comp": c.compression,                      # Compression codec (SNAPPY)
        "encodings": ",".join(c.encodings),         # Encodings used (Dictionary + RLE + PLAIN)
        "dict_off": c.dictionary_page_offset,       # Byte offset where the dictionary page starts
        "data_off": c.data_page_offset,             # Byte offset where the compressed data pages start
        "raw_B": c.total_uncompressed_size,         # Uncompressed size of this column chunk (bytes)
        "comp_B": c.total_compressed_size,          # Compressed size on disk (bytes)
        "nulls": st.null_count if st else None,     # Missing values count stored in footer
        "min_stat": repr(st.min) if st and st.has_min_max else None,  # Min value in Row Group
        "max_stat": repr(st.max) if st and st.has_min_max else None,  # Max value in Row Group
    })

df_chunks = pd.DataFrame(chunk_rows)

# Verify the exact binary byte budget:
# 4B header ("PAR1") + all compressed column chunks + Thrift footer + 8B tail (4B len + 4B "PAR1")
total_payload_b = df_chunks["comp_B"].sum()
assert 4 + total_payload_b + footer_len + 8 == len(raw_bytes), "Byte map mismatch!"

print(f"Exact binary map: 4B (PAR1) + {total_payload_b:,}B (Column Chunks) + {footer_len:,}B (Footer) + 8B (len+PAR1) = {len(raw_bytes):,}B")
print("\nColumn Chunk Metadata & Footer Statistics (Row Group 0):")
print(df_chunks.to_string(index=False))


Exact binary map: 4B (PAR1) + 254,029B (Column Chunks) + 6,985B (Footer) + 8B (len+PAR1) = 261,026B

Column Chunk Metadata & Footer Statistics (Row Group 0):
                  column  phys_type   comp                encodings  dict_off  data_off  raw_B  comp_B  nulls                           min_stat                           max_stat
               timestamp BYTE_ARRAY SNAPPY PLAIN,RLE,RLE_DICTIONARY         4     25054 170373   57452      0              '2026-10-01 06:00:00'              '2099-01-01 12:00:00'
                 line_id BYTE_ARRAY SNAPPY PLAIN,RLE,RLE_DICTIONARY     57456     57529   8931    3662      0                           'LINE_A'                           'line_c'
           bottle_format BYTE_ARRAY SNAPPY PLAIN,RLE,RLE_DICTIONARY     61118     61167   5778    1714      0                        'GLASS_075'                           'PET_15'
          ambient_temp_C     DOUBLE SNAPPY PLAIN,RLE,RLE_DICTIONARY     62832     65260  33800   31598      0             

### 3.3 Demo: Projection Pushdown (`columns=[...]`)
Reading only `["line_id", "filler_pressure_bar"]` uses the byte offsets in the Parquet footer to jump directly to those 2 columns and skip >80% of Parquet column chunk bytes (and >96% compared to scanning the raw CSV!).

In [4]:
# Ready-to-run Demo: Projection Pushdown (skip unused columns using Footer byte offsets)
selected_cols = ["line_id", "filler_pressure_bar"]
csv_bytes = (BRONZE_RAW / "line_telemetry.csv").stat().st_size if (BRONZE_RAW / "line_telemetry.csv").exists() else 0

# PyArrow reads ONLY the 2 requested columns instead of all 12
tbl_proj = pq.read_table(pq_path, columns=selected_cols)

# Sum the compressed byte sizes of only those 2 Column Chunks from Row Group 0 metadata
proj_bytes = sum(
    rg.column(i).total_compressed_size
    for i in range(rg.num_columns)
    if rg.column(i).path_in_schema in selected_cols
)
print(f"Projection Pushdown ({selected_cols}):")
print(f"  Raw CSV scan: {csv_bytes:,} B -> Full Parquet chunks: {total_payload_b:,} B -> Projected 2 cols: {proj_bytes:,} B ({100 * (1 - proj_bytes / total_payload_b):.1f}% Parquet I/O skipped!)")
if csv_bytes > 0:
    pct_csv_read = 100 * proj_bytes / csv_bytes
    print(f"  Efficiency vs Raw CSV: read only {pct_csv_read:.2f}% of CSV bytes ({100 - pct_csv_read:.2f}% I/O skipped — {csv_bytes / proj_bytes:.1f}x less data read!)")
print(f"  Projected table shape: {tbl_proj.num_rows:,} rows x {tbl_proj.num_columns} cols")


Projection Pushdown (['line_id', 'filler_pressure_bar']):
  Raw CSV scan: 1,500,439 B -> Full Parquet chunks: 254,029 B -> Projected 2 cols: 31,452 B (87.6% Parquet I/O skipped!)
  Efficiency vs Raw CSV: read only 2.10% of CSV bytes (97.90% I/O skipped — 47.7x less data read!)
  Projected table shape: 23,164 rows x 2 cols


### 3.4 Demo: Multi-Row-Group Predicate Pushdown (`filters=[...]`)
When a Parquet file spans multiple Row Groups (`row_group_size=6000`), the query engine inspects each Row Group's footer `[min, max]` and **skips entire Row Groups** whose range cannot match! Notice also how uncleaned `'line_c'` lands in Row Group 3 (`'l' > 'L'` in ASCII), proving why Section 5 string hygiene matters for filtering.

In [5]:
# Ready-to-run Demo: Multi-Row-Group Predicate Pushdown (skip whole Row Groups using [min, max] stats)
# Write an in-memory Parquet buffer sorted by line_id and split into 4 Row Groups
sink = pa.BufferOutputStream()
pq.write_table(
    pa.Table.from_pandas(df_telem.sort_values("line_id"), preserve_index=False),
    sink,
    row_group_size=6000,   # Splits 23,164 rows into 4 Row Groups (~6,000 rows each)
    compression="snappy",
)
multi_buf = sink.getvalue()
pf_multi = pq.ParquetFile(pa.BufferReader(multi_buf))

# Apply a filter pushdown: PyArrow checks each Row Group's [min, max] footer stats BEFORE reading data
tbl_pred = pq.read_table(
    pa.BufferReader(multi_buf),
    columns=["timestamp", "line_id", "filler_pressure_bar"],
    filters=[("line_id", "=", "LINE_C")],
)

print(f"Multi-Row-Group Stats ({pf_multi.metadata.num_row_groups} Row Groups, sorted by line_id):")
for r in range(pf_multi.metadata.num_row_groups):
    rg_r = pf_multi.metadata.row_group(r)
    st_m = rg_r.column(1).statistics  # column 1 is line_id
    # If "LINE_C" falls outside [min, max], the entire Row Group is skipped without decompression
    can_skip = not (st_m.min <= "LINE_C" <= st_m.max)
    print(f"  RG {r}: rows={rg_r.num_rows:>4} | line_id=[{st_m.min!r}, {st_m.max!r}] -> {'SKIPPED by footer!' if can_skip else 'CANDIDATE'}")
print(f"Predicate Pushdown result ('LINE_C'): {tbl_pred.num_rows:,} rows (notice uncleaned 'line_c' in RG 3 was missed!)")


Multi-Row-Group Stats (4 Row Groups, sorted by line_id):
  RG 0: rows=6000 | line_id=['LINE_A', 'LINE_B'] -> SKIPPED by footer!
  RG 1: rows=6000 | line_id=['LINE_B', 'LINE_C'] -> CANDIDATE
  RG 2: rows=6000 | line_id=['LINE_C', 'LINE_D'] -> CANDIDATE
  RG 3: rows=5164 | line_id=['LINE_D', 'line_c'] -> SKIPPED by footer!
Predicate Pushdown result ('LINE_C'): 5,106 rows (notice uncleaned 'line_c' in RG 3 was missed!)


## 🔍 4. File 1 — `bronze_telemetry.parquet` (PLC Telemetry: One Issue at a Time)
**Critical Rule (`Bronze Immutability`):** Never overwrite files in `bronze_staged/`. We inspect `df_telem` first (**Discovery**), then fix **one structural issue at a time** in memory (**Solution**) before moving on to File 2 (`df_stop`).

### 4.1 Discovery — Dirty Line Keys (`line_id`)
**The Problem:** Industrial PLC logs often suffer from trailing spaces (`'Line_A '`), inconsistent casing (`'line_c'`), or legacy aliases (`'LN-B'`). Before cleaning anything, compare the distinct `line_id` values in `df_telem` against the canonical ERP master list (`df_lines["line_id"]`).

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `Series.astype(str).str.strip().str.upper()` | Remove spaces & uppercase text | `df_lines["line_id"].astype(str).str.strip().str.upper()` |
> | `Series.value_counts(dropna=False)` | Count every unique value (including `NaN`) | `df_telem["line_id"].value_counts(dropna=False)` |
> | `Series.isin(values)` | Boolean mask: `True` if value is in a list/set | `df_telem["line_id"].isin(["Line_A ", "LN-B", "line_c"])` |

In [6]:
# Canonical ERP bottling line set (e.g. {'LINE_A', 'LINE_B', 'LINE_C', 'LINE_D'})
valid_lines = set(df_lines["line_id"].astype(str).str.strip().str.upper())
raw_line_counts = df_telem["line_id"].value_counts(dropna=False)
alias_tel_count = int(df_telem["line_id"].isin(["Line_A ", "LN-B", "line_c"]).sum())

print("Canonical ERP lines:", sorted(valid_lines))
print("\nRaw PLC line_id counts (spot 'Line_A ', 'LN-B', 'line_c' & orphan 'LINE_Z'):")
for k, cnt in raw_line_counts.items():
    flag = "  <-- CLEAN" if k in valid_lines else "  <-- DIRTY / UNKNOWN"
    print(f"  {repr(k):<14} -> {cnt:>6,} rows{flag}")
print(f"\nTotal telemetry rows needing string/alias normalization: {alias_tel_count:,}")


Canonical ERP lines: ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D']

Raw PLC line_id counts (spot 'Line_A ', 'LN-B', 'line_c' & orphan 'LINE_Z'):
  'LINE_D'       ->  5,784 rows  <-- CLEAN
  'LINE_C'       ->  5,106 rows  <-- CLEAN
  'LINE_B'       ->  4,395 rows  <-- CLEAN
  'LINE_A'       ->  4,172 rows  <-- CLEAN
  'Line_A '      ->  1,620 rows  <-- DIRTY / UNKNOWN
  'LN-B'         ->  1,390 rows  <-- DIRTY / UNKNOWN
  'line_c'       ->    692 rows  <-- DIRTY / UNKNOWN
  'LINE_Z'       ->      5 rows  <-- DIRTY / UNKNOWN

Total telemetry rows needing string/alias normalization: 3,702


### 4.2 Solution — Normalize Telemetry `line_id` (String Hygiene & Alias Map)
**The Fix:** Strip whitespace, convert to uppercase, and translate legacy aliases (`'LN-B'` -> `'LINE_B'`) deterministically via a reusable `clean_line_code(val)` function. Notice that unknown ghost line `'LINE_Z'` is **kept as `'LINE_Z'`** (not deleted here) so Lab 04 can audit and quarantine it!

> **Pandas & Python Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `dict.get(key, default)` | Look up `key`; return `default` if absent | `LINE_ALIASES.get(s, s)` |
> | `Series.apply(func)` | Apply a Python function element-by-element | `df_telem["line_id"] = df_telem["line_id"].apply(clean_line_code)` |
> | `Series.unique()` | Return array of distinct values for verification | `sorted(df_telem["line_id"].unique())` |

In [7]:
# Canonical alias lookup table mapping legacy variants -> standard ERP bottling line IDs
LINE_ALIASES = {
    "LN-A": "LINE_A", "LINE-A": "LINE_A", "L_A": "LINE_A",
    "LN-B": "LINE_B", "LINE-B": "LINE_B", "L_B": "LINE_B",
    "LN-C": "LINE_C", "LINE-C": "LINE_C", "L_C": "LINE_C",
    "LN-D": "LINE_D", "LINE-D": "LINE_D", "L_D": "LINE_D",
}

def clean_line_code(val):
    if pd.isna(val):                              # Guard against missing/NaN values
        return "UNKNOWN"
    s = str(val).strip().upper()                  # Remove spaces ('Line_A ') & standardize case ('line_c')
    return LINE_ALIASES.get(s, s)                 # Map alias ('LN-B' -> 'LINE_B') or keep as-is ('LINE_Z')

print("BEFORE -> telemetry line_id:", sorted(df_telem["line_id"].dropna().unique()))

df_lines["line_id"] = df_lines["line_id"].apply(clean_line_code)
df_telem["line_id"] = df_telem["line_id"].apply(clean_line_code)

print("AFTER  -> telemetry line_id:", sorted(df_telem["line_id"].dropna().unique()))
assert df_telem["line_id"].str.contains(" ").sum() == 0, "Whitespace still present in line_id!" 


BEFORE -> telemetry line_id: ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D', 'LINE_Z', 'LN-B', 'Line_A ', 'line_c']
AFTER  -> telemetry line_id: ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D', 'LINE_Z']


### 4.3 Discovery — Why Text Timestamps Break Pipelines (`timestamp`)
**The Problem:** In Bronze, `df_telem["timestamp"]` was intentionally left as raw **`string` (`object`)** text. Before parsing it, run three short experiments proving why leaving timestamps as text breaks analytical pipelines:
1. **Broken Time Math:** Subtracting `pd.Timedelta(hours=6)` raises `TypeError` (and Lab 04's `.interpolate(method='time')` fails).
2. **Silent Sorting Bugs:** Strings compare ASCII characters left-to-right, NOT chronology (`'09/10/2026' > '01/11/2026'` is `True`; `'2026-10-01T06:00:00' > '2026-10-01 07:00:00'` is `True` because `'T' > ' '`!).
3. **~8.5× Extra RAM:** Native `datetime64[ns]` stores compact 8-byte integers instead of Python string objects.

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `Series.memory_usage(deep=True)` | Exact RAM footprint in bytes (inspects Python strings) | `df_telem["timestamp"].memory_usage(deep=True)` |
> | `pd.to_datetime(s, errors="coerce")` | Parse text -> `datetime64[ns]` (invalid text -> `NaT`) | `pd.to_datetime(df_telem["timestamp"], errors="coerce")` |
> | `pd.Timedelta(hours=6)` | Time duration for timestamp arithmetic | `ts - pd.Timedelta(hours=6)` |

In [8]:
# Experiment 1: Temporal arithmetic fails on raw string timestamps
try:
    _ = df_telem["timestamp"].head(3) - pd.Timedelta(hours=6)
except TypeError as err:
    print(f"[Experiment 1] String timestamp math FAILED as expected -> TypeError: {err}")

# Experiment 2: Silent alphabetical vs chronological ordering bugs
eu_cmp_str = "09/10/2026" > "01/11/2026"
iso_cmp_str = "2026-10-01T06:00:00" > "2026-10-01 07:00:00"  # 'T' (ASCII 84) > ' ' (ASCII 32)!
num_cmp_str = "9" > "10"
print(f"[Experiment 2] String traps -> '09/10/2026' > '01/11/2026' is {eu_cmp_str}, '2026-10-01T06:00:00' > '2026-10-01 07:00:00' is {iso_cmp_str}, '9' > '10' is {num_cmp_str}!")

# Experiment 3: RAM usage comparison (object/str vs native datetime64[ns])
ram_str_bytes = df_telem["timestamp"].memory_usage(deep=True)
ram_dt_bytes = pd.to_datetime(df_telem["timestamp"], errors="coerce").memory_usage(deep=True)
print(f"[Experiment 3] RAM for 'timestamp' -> object/str: {ram_str_bytes:,} B vs datetime64[ns]: {ram_dt_bytes:,} B ({ram_str_bytes / ram_dt_bytes:.2f}x smaller!)")


[Experiment 1] String timestamp math FAILED as expected -> TypeError: unsupported operand type(s) for -: 'str' and 'Timedelta'
[Experiment 2] String traps -> '09/10/2026' > '01/11/2026' is True, '2026-10-01T06:00:00' > '2026-10-01 07:00:00' is True, '9' > '10' is True!
[Experiment 3] RAM for 'timestamp' -> object/str: 1,575,284 B vs datetime64[ns]: 185,444 B (8.49x smaller!)


### 4.4 Solution — Parse `timestamp` & Fix Night-Shift `S3` Midnight Rollover
**The Problem & Fix:** Now parse `df_telem["timestamp"]` into native `datetime64[ns]`. Then assign the factory shift (`S1`: `06–14`, `S2`: `14–22`, `S3`: `22–06`) and compute **`operational_shift_date = (ts - 6 hours).dt.date`** so night-shift readings between `00:00` and `05:45` stay attached to the shift's starting operational date — stored as true **`datetime.date`** (Parquet `DATE` / `date32[day]`, **never** `.astype(str)`!).

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `pd.to_datetime(s, errors="coerce")` | Convert string column to `datetime64[ns]` | `df_telem["timestamp"] = pd.to_datetime(df_telem["timestamp"], errors="coerce")` |
> | `Series.dt.hour` | Extract hour `0..23` from a datetime Series | `ts.dt.hour.fillna(-1).astype(int)` |
> | `(ts - pd.Timedelta(hours=6)).dt.date` | Shift timestamps back 6h and extract native `DATE` | `(df_telem["timestamp"] - pd.Timedelta(hours=6)).dt.date` |

In [9]:
# Reusable helper: map each timestamp's hour to S1 (06-14), S2 (14-22), or S3 (22-06)
def assign_shift_code(ts: pd.Series) -> pd.Series:
    return pd.Series(
        ["S1" if 6 <= h < 14 else ("S2" if 14 <= h < 22 else "S3") for h in ts.dt.hour.fillna(-1).astype(int)],
        index=ts.index,
    )

# 1. Parse PLC telemetry timestamps from text into native datetime64[ns]
df_telem["timestamp"] = pd.to_datetime(df_telem["timestamp"], errors="coerce")

# 2. Assign 8-hour shift_code and operational_shift_date (rolling 00:00-05:59 back to previous day)
df_telem["shift_code"] = assign_shift_code(df_telem["timestamp"])
df_telem["operational_shift_date"] = (df_telem["timestamp"] - pd.Timedelta(hours=6)).dt.date

print("Telemetry timestamp dtype:", df_telem["timestamp"].dtype)
sample_s3 = df_telem[df_telem["timestamp"].dt.hour == 2][["timestamp", "shift_code", "operational_shift_date"]].head(3)
print("\nSample S3 post-midnight rollover check (02:00 belongs to previous operational date):")
print(sample_s3.to_string(index=False))


Telemetry timestamp dtype: datetime64[ns]

Sample S3 post-midnight rollover check (02:00 belongs to previous operational date):
          timestamp shift_code operational_shift_date
2026-10-02 02:00:00         S3             2026-10-01
2026-10-02 02:00:00         S3             2026-10-01
2026-10-02 02:00:00         S3             2026-10-01


### 4.5 Discovery — MQTT Duplicate Telemetry Rows
**The Problem:** Industrial MQTT brokers occasionally resend the same packet on retry, appending duplicate rows at the end of the file for the same `(timestamp, line_id, bottle_format)`. Inspecting duplicates **after** `line_id` normalization catches them regardless of raw spelling — and **sorting by `KEY_COLS` before `.head(4)`** puts each original row and its retry right next to each other (keeping the row index visible so you can see the original row vs. the appended retry!).

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `df.duplicated(subset=KEY_COLS)` | Boolean mask marking duplicate rows after the first occurrence | `df_telem.duplicated(subset=KEY_COLS).sum()` |
> | `df.duplicated(subset=KEY_COLS, keep=False)` | Mark **all** copies of every duplicated key | `df_telem.loc[df_telem.duplicated(subset=KEY_COLS, keep=False)]` |
> | `.sort_values(KEY_COLS)` | Group matching duplicate keys side-by-side before `.head()` | `all_dupes.sort_values(KEY_COLS).groupby(KEY_COLS, sort=False).head(2).head(4)` |

In [10]:
KEY_COLS = ["timestamp", "line_id", "bottle_format"]

# Discovery (Read-Only): Count & display actual duplicate pairs side-by-side with original row indices
dupes_count = int(df_telem.duplicated(subset=KEY_COLS).sum())
print(f"[Discovery] Found {dupes_count:,} duplicate telemetry rows on {KEY_COLS}.")

all_dupes = df_telem.loc[df_telem.duplicated(subset=KEY_COLS, keep=False), [*KEY_COLS, "bottled_units"]]
sample_dupes = all_dupes.sort_values(KEY_COLS).groupby(KEY_COLS, sort=False).head(2).head(4)
print("\nSample duplicate pairs (leftmost column = original file row index):")
print(sample_dupes.to_string())


[Discovery] Found 114 duplicate telemetry rows on ['timestamp', 'line_id', 'bottle_format'].

Sample duplicate pairs (leftmost column = original file row index):
                timestamp line_id bottle_format  bottled_units
10    2026-10-01 06:30:00  LINE_C        PET_15            934
23081 2026-10-01 06:30:00  LINE_C        PET_15            934
17    2026-10-01 07:00:00  LINE_B     GLASS_075            809
23121 2026-10-01 07:00:00  LINE_B     GLASS_075            809


### 4.6 Solution — Deduplicate Telemetry Rows (`keep='last'`)
**The Fix:** Remove duplicate rows on `KEY_COLS`, keeping the latest retry (`keep="last"` follows file row order, so row `23081` is kept over row `10`), and chain `.copy()` to prevent `SettingWithCopyWarning` in later steps.

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `df.drop_duplicates(subset=KEY_COLS, keep="last").copy()` | Drop duplicates keeping the last occurrence & return a clean copy | `df_telem = df_telem.drop_duplicates(subset=KEY_COLS, keep="last").copy()` |

In [11]:
before_tel = len(df_telem)
df_telem = df_telem.drop_duplicates(subset=KEY_COLS, keep="last").copy()
remaining = int(df_telem.duplicated(subset=KEY_COLS).sum())
assert remaining == 0, "Duplicates still remain in df_telem!"
print(f"[Solution] Removed {before_tel - len(df_telem):,} duplicates -> kept {len(df_telem):,} rows (remaining duplicates: {remaining}).")


[Solution] Removed 114 duplicates -> kept 23,050 rows (remaining duplicates: 0).


### 4.7 Discovery — Telemetry Anomalies Deferred to Lab 04 (Read-Only Triage)
**The Problem:** Structural hygiene on `df_telem` is done, **but bad sensor values still exist** (impossible filler pressure `-99.0`/`85.0 bar`, missing `pasteurizer_temp_C`, `2099` future dates, `rejected_units > bottled_units`, and orphan `LINE_Z`). Do **NOT** delete them here — quantify them now so Lab 04 can route them through DAMA Quarantine and Time Interpolation!

> **Pandas Syntax Reference:**
> | Method / Operator | Purpose | Example |
> | :--- | :--- | :--- |
> | `(cond_a) \| (cond_b)` | Combine boolean masks with element-wise OR (always use parentheses!) | `(df_telem["filler_pressure_bar"] < 0) \| (df_telem["filler_pressure_bar"] > 12)` |
> | `Series.isna().sum()` | Count missing (`NaN` / `NaT`) entries | `df_telem["pasteurizer_temp_C"].isna().sum()` |
> | `~Series.isin(valid_lines)` | Negate a mask to find unknown/orphan keys | `(~df_telem["line_id"].isin(valid_lines)).sum()` |

In [12]:
bad_pressure_count = int(((df_telem["filler_pressure_bar"] < 0) | (df_telem["filler_pressure_bar"] > 12)).sum())
null_temp_count = int(df_telem["pasteurizer_temp_C"].isna().sum())
orphan_tel_count = int((~df_telem["line_id"].isin(valid_lines)).sum())
future_tel_count = int((df_telem["timestamp"] > "2027-01-01").sum())
bad_reject_count = int(
    ((df_telem["bottled_units"] < 0) | (df_telem["rejected_units"] < 0) | (df_telem["rejected_units"] > df_telem["bottled_units"])).sum()
)

print("=== FILE 1 (df_telem) READ-ONLY ANOMALIES FOR LAB 04 ===")
print(f"  Orphan line_id (LINE_Z)               : {orphan_tel_count}")
print(f"  Future timestamps (> 2027-01-01)      : {future_tel_count}")
print(f"  Out-of-bounds pressure (<0 or >12 bar): {bad_pressure_count}")
print(f"  Impossible unit counts                : {bad_reject_count}")
print(f"  Missing pasteurizer_temp_C (NaN)      : {null_temp_count}  (to interpolate in Lab 04, NOT drop!)")


=== FILE 1 (df_telem) READ-ONLY ANOMALIES FOR LAB 04 ===
  Orphan line_id (LINE_Z)               : 5
  Future timestamps (> 2027-01-01)      : 1
  Out-of-bounds pressure (<0 or >12 bar): 40
  Impossible unit counts                : 4
  Missing pasteurizer_temp_C (NaN)      : 120  (to interpolate in Lab 04, NOT drop!)


## 📋 5. File 2 — `bronze_stoppages.parquet` (CMMS Stoppage Logs: Discovery → Solution)
Now that File 1 (`df_telem`) is completely handled, we apply the **exact same Discovery → Solution workflow** to File 2 (`df_stop`, 292 CMMS stoppage logs) in isolation.

### 5.1 Discovery — Stoppage Keys, Category Drift, Duplicates & Phantom Log
**The Problem:** Inspect `df_stop` for four concrete issues:
1. Dirty `line_code` values (`'Line_A '`, `'LN-B'`) and orphan `'LINE_Z'`.
2. Casing drift in `stoppage_type` (`'unplanned'` vs `'Unplanned'` vs `'Preventive'`).
3. Duplicate `log_id` records.
4. Phantom future log `STOP-20269999` (`2099-01-01`, `LINE_Z`) — to flag now and quarantine in Lab 04.

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `Series.value_counts(dropna=False)` | Count distinct category values | `df_stop["stoppage_type"].value_counts(dropna=False)` |
> | `df.duplicated(subset=["log_id"]).sum()` | Count duplicate stoppage IDs | `df_stop.duplicated(subset=["log_id"]).sum()` |
> | `df.loc[mask, cols]` | Display only flagged rows & selected columns | `df_stop.loc[stop_bad_mask, ["log_id", "event_date", "line_code"]]` |

In [13]:
stop_type_counts = df_stop["stoppage_type"].value_counts(dropna=False)
stop_dupes_count = int(df_stop.duplicated(subset=["log_id"]).sum())

clean_stop_codes = df_stop["line_code"].astype(str).str.strip().str.upper()
stop_ts = pd.to_datetime(df_stop["event_date"], errors="coerce")
is_future_or_invalid = stop_ts.isna() | (stop_ts > "2027-01-01")
stop_future_orphan_count = int((stop_ts > "2027-01-01").sum())
stop_unplanned_lower_count = int((df_stop["stoppage_type"] == "unplanned").sum())

print("1. Raw stoppage line_code values:", sorted(df_stop["line_code"].dropna().unique()))
print("\n2. Raw stoppage_type counts (notice 'unplanned' vs 'Unplanned'):")
print(stop_type_counts.to_string())
print(f"\n3. Duplicate log_id rows: {stop_dupes_count} | Defensive downtime_min <= 0: {(df_stop['downtime_min'] <= 0).sum()}")
print("\n4. Phantom / future stoppage log (to keep for Lab 04 Quarantine):")
print(df_stop.loc[is_future_or_invalid | (clean_stop_codes == "LINE_Z"), ["log_id", "event_date", "line_code", "stoppage_type", "root_cause", "downtime_min"]].to_string(index=False))


1. Raw stoppage line_code values: ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D', 'LINE_Z', 'LN-B', 'Line_A ']

2. Raw stoppage_type counts (notice 'unplanned' vs 'Unplanned'):
stoppage_type
Unplanned     177
Preventive     79
unplanned      36

3. Duplicate log_id rows: 2 | Defensive downtime_min <= 0: 0

4. Phantom / future stoppage log (to keep for Lab 04 Quarantine):
       log_id          event_date line_code stoppage_type  root_cause  downtime_min
STOP-20269999 2099-01-01 03:00:00    LINE_Z     Unplanned Phantom Log            60


### 5.2 Solution — Clean `df_stop` (Keys, Categories, `event_date`, Dedup & `S3` Shift Date)
**The Fix:** Reuse `clean_line_code` and `assign_shift_code` from Section 4, standardize `stoppage_type` (`'Unplanned'`, `'Preventive'`, `'Corrective'`), parse `event_date` to `datetime64[ns]`, deduplicate on `log_id` (`keep='last'`), and compute `shift_code` + `operational_shift_date`.

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `Series.apply(func)` | Standardize string columns element-wise | `df_stop["stoppage_type"] = df_stop["stoppage_type"].apply(standardize_stoppage_type)` |
> | `pd.to_datetime(s, errors="coerce")` | Parse `event_date` to native `datetime64[ns]` | `df_stop["event_date"] = pd.to_datetime(df_stop["event_date"], errors="coerce")` |
> | `df.drop_duplicates(subset=["log_id"], keep="last").copy()` | Remove duplicate stoppage logs safely | `df_stop = df_stop.drop_duplicates(subset=["log_id"], keep="last").copy()` |

In [14]:
def standardize_stoppage_type(val):
    s = str(val).strip().lower()
    if "prevent" in s or "pm" in s:
        return "Preventive"
    if "correct" in s:
        return "Corrective"
    return "Unplanned"

# 1. Normalize line_code (reusing clean_line_code) and stoppage_type
df_stop["line_code"] = df_stop["line_code"].apply(clean_line_code)
df_stop["stoppage_type"] = df_stop["stoppage_type"].apply(standardize_stoppage_type)

# 2. Parse event_date into native datetime64[ns]
df_stop["event_date"] = pd.to_datetime(df_stop["event_date"], errors="coerce")

# 3. Deduplicate on log_id (keep='last' + .copy())
before_stop = len(df_stop)
df_stop = df_stop.drop_duplicates(subset=["log_id"], keep="last").copy()

# 4. Assign shift_code & operational_shift_date (reusing assign_shift_code)
df_stop["shift_code"] = assign_shift_code(df_stop["event_date"])
df_stop["operational_shift_date"] = (df_stop["event_date"] - pd.Timedelta(hours=6)).dt.date

print("Clean stoppage line_code    :", sorted(df_stop["line_code"].unique()))
print("Clean stoppage_type counts  :", df_stop["stoppage_type"].value_counts().to_dict())
print(f"Stoppage dedup removed {before_stop - len(df_stop):,} duplicate logs (kept {len(df_stop):,}; STOP-20269999 preserved for Lab 04).")


Clean stoppage line_code    : ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D', 'LINE_Z']
Clean stoppage_type counts  : {'Unplanned': 212, 'Preventive': 78}
Stoppage dedup removed 2 duplicate logs (kept 290; STOP-20269999 preserved for Lab 04).


## 📊 6. Cross-Table Triage — Bronze DAMA DQ Scorecard & Intermediate Export
### 6.1 Compile the Bronze DAMA DQ Scorecard
**The Goal:** Combine our discoveries from File 1 (`df_telem`, Section 4) and File 2 (`df_stop`, Section 5) into one audit scorecard. Notice the key distinction for Lab 04:
- **Unrecoverable row errors** (`DQR-101, 102, 104, 105` — 11 rows) → **Quarantine & remove row**.
- **Recoverable single-sensor glitches on valid production rows** (`DQR-103` pressure spikes + `DQR-TEMP` null temps) → **Audit-log, nullify & time-interpolate** so we never lose real bottled production!

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `pd.DataFrame([{...}, ...])` | Build a structured summary table from a list of dicts | `pd.DataFrame([{"Rule_ID": "DQR-101", "Count": 5}])` |

In [15]:
dq_scorecard = pd.DataFrame([
    {"Rule_ID": "DQR-KEY",  "Source": "PLC + CMMS", "Anomaly": "Whitespace/casing/alias drift ('Line_A ', 'LN-B', 'unplanned')", "Count": alias_tel_count + stop_unplanned_lower_count, "Planned_Action": "Lab 03 Done — Normalized"},
    {"Rule_ID": "DQR-107",  "Source": "PLC + CMMS", "Anomaly": "Duplicate rows (114 telemetry + 2 stoppage logs)",              "Count": dupes_count + stop_dupes_count,               "Planned_Action": "Lab 03 Done — Deduplicated (keep='last')"},
    {"Rule_ID": "DQR-101",  "Source": "PLC CSV",    "Anomaly": "Orphan ghost asset (LINE_Z)",                                   "Count": orphan_tel_count,                             "Planned_Action": "Lab 04 Quarantine Row (BLOCKER)"},
    {"Rule_ID": "DQR-102",  "Source": "PLC CSV",    "Anomaly": "Future 2099-* telemetry timestamp",                             "Count": future_tel_count,                             "Planned_Action": "Lab 04 Quarantine Row"},
    {"Rule_ID": "DQR-104",  "Source": "PLC CSV",    "Anomaly": "Negative or rejected_units > bottled_units",                    "Count": bad_reject_count,                             "Planned_Action": "Lab 04 Quarantine Row"},
    {"Rule_ID": "DQR-105",  "Source": "CMMS JSON",  "Anomaly": "Phantom/future stoppage log (STOP-20269999)",                   "Count": stop_future_orphan_count,                     "Planned_Action": "Lab 04 Quarantine Row"},
    {"Rule_ID": "DQR-103",  "Source": "PLC CSV",    "Anomaly": "Impossible filler_pressure_bar (-99.0 / 85.0 bar)",             "Count": bad_pressure_count,                           "Planned_Action": "Lab 04 Log, Nullify & Time Interpolate (Keep Bottles!)"},
    {"Rule_ID": "DQR-TEMP", "Source": "PLC CSV",    "Anomaly": "Missing pasteurizer_temp_C (NaN)",                              "Count": null_temp_count,                              "Planned_Action": "Lab 04 Time Interpolate (NO dropna!)"},
])
print("=== BRONZE DATA QUALITY SCORECARD (TRIAGE PLAN) ===")
print(dq_scorecard.to_string(index=False))


=== BRONZE DATA QUALITY SCORECARD (TRIAGE PLAN) ===
 Rule_ID     Source                                                        Anomaly  Count                                         Planned_Action
 DQR-KEY PLC + CMMS Whitespace/casing/alias drift ('Line_A ', 'LN-B', 'unplanned')   3738                               Lab 03 Done — Normalized
 DQR-107 PLC + CMMS               Duplicate rows (114 telemetry + 2 stoppage logs)    116               Lab 03 Done — Deduplicated (keep='last')
 DQR-101    PLC CSV                                    Orphan ghost asset (LINE_Z)      5                        Lab 04 Quarantine Row (BLOCKER)
 DQR-102    PLC CSV                              Future 2099-* telemetry timestamp      1                                  Lab 04 Quarantine Row
 DQR-104    PLC CSV                     Negative or rejected_units > bottled_units      4                                  Lab 04 Quarantine Row
 DQR-105  CMMS JSON                    Phantom/future stoppage log (STOP-20269

### 6.2 Save Intermediate Tables to `silver_part1/*.parquet` (Handoff to Lab 04)
**The Goal:** Persist our structurally cleaned tables (`silver_part1/*.parquet`) so **Lab 04** can pick up right here to run Statistical Outlier Analysis (`IQR` vs. `3-Sigma`), DAMA Quarantine & Sensor Repair, Time-Series Interpolation, and the **Final Silver Schema Contract & DuckDB Audit**.

> **Pandas Syntax Reference:**
> | Method | Purpose | Example |
> | :--- | :--- | :--- |
> | `df.to_parquet(path, index=False, compression="snappy")` | Save DataFrame to Snappy Parquet without leaking `__index_level_0__` | `df_out.to_parquet(SILVER_P1 / fname, index=False, compression="snappy")` |

In [16]:
assert df_telem["line_id"].str.contains(" ").sum() == 0, "Whitespace remains in line_id!"
assert set(df_stop["stoppage_type"]).issubset({"Unplanned", "Preventive", "Corrective"}), "Unstandardized stoppage types!"

outputs = {
    "silver_part1_telemetry.parquet": df_telem,
    "silver_part1_stoppages.parquet": df_stop,
    "silver_lines.parquet": df_lines,
    "silver_shifts.parquet": df_shifts,
    "silver_bottle_formats.parquet": df_formats,
}
for fname, df_out in outputs.items():
    df_out.to_parquet(SILVER_P1 / fname, index=False, compression="snappy")
    print(f"  [SAVED] {fname:<34} ({len(df_out):>6,} rows)")

print(f"\nIntermediate Silver Part 1 Parquets ready in: {SILVER_P1}")
print("NOTE: LINE_Z, 2099 future dates, -99/85 bar sensor spikes & NaN temps are intentionally preserved for Lab 04!")


  [SAVED] silver_part1_telemetry.parquet     (23,050 rows)
  [SAVED] silver_part1_stoppages.parquet     (   290 rows)
  [SAVED] silver_lines.parquet               (     4 rows)
  [SAVED] silver_shifts.parquet              (     3 rows)
  [SAVED] silver_bottle_formats.parquet      (     3 rows)

Intermediate Silver Part 1 Parquets ready in: <REPO_ROOT>/03_labs_student_starters/sandbox_outputs/case0_europack_bottling/silver_part1
NOTE: LINE_Z, 2099 future dates, -99/85 bar sensor spikes & NaN temps are intentionally preserved for Lab 04!
